# 03 · DataFrames with dflib

Hardwood streams. **dflib** is the other half: a DataFrame that you *shape* —
select, filter, derive, group, sort. This is the language of everyday data work,
and it is plain Java.

The catalog is loaded once, in memory, on purpose: 21 207 × 52 is small.

In [0]:
addDependency("org.dflib:dflib:2.0.0-M7");
addDependency("org.dflib:dflib-parquet:2.0.0-M7");

import org.dflib.DataFrame;
import org.dflib.parquet.Parquet;
import static org.dflib.Exp.*;
import java.nio.file.Files;
import java.nio.file.Path;

String num(double v) { return String.format(java.util.Locale.ROOT, "%,.0f", v); }
String dec(double v, int p) { return String.format(java.util.Locale.ROOT, "%." + p + "f", v); }

// The notebook may live in a sub-folder (exercises/, bonus/, ...), so walk up
// from the notebook's own folder to find the repository's data/ directory.
Path dataset() {
    for (Path p = cwd.toAbsolutePath(); p != null; p = p.getParent()) {
        Path candidate = p.resolve("data/celestrak_gp_catalog.parquet");
        if (Files.isRegularFile(candidate)) return candidate;
    }
    throw new IllegalStateException("data/celestrak_gp_catalog.parquet not found above " + cwd);
}

var df = Parquet.loader().load(dataset());
println("loaded %s rows x %d columns", num(df.height()), df.width());

## 1 · Select

Keep the columns you care about. `cols(...)` picks them, `select()` turns the
selection back into a frame.

In [0]:
var sel = df.cols("object_name", "orbit_class", "apogee_km").select();
System.out.println(sel.head(5));

## 2 · Filter by text

Conditions are built from expressions. `$str("col").eq("GEO")` gives a
`Condition`; `rows(...)` keeps the matching rows.

In [0]:
var geo = df.rows($str("orbit_class").eq("GEO")).select();
println("GEO objects: %s", num(geo.height()));

## 3 · Filter by number

Numeric columns use `$double(...)` and the usual comparisons. Highly eccentric
orbits are where the interesting objects hide.

In [0]:
var eccentric = df.rows($double("eccentricity").gt(0.25)).select();
println("eccentricity > 0.25: %s objects", num(eccentric.height()));

## 4 · Derive and aggregate

This is where a DataFrame earns its keep: compute something new *and* summarise
it in one breath — here the mean gap between apogee and perigee, per orbit class.
`$double("apogee_km").sub($double("perigee_km"))` is a derived expression, and
`.avg().as("...")` names the result.

In [0]:
System.out.println(
    df.group("orbit_class").agg(
        $col("orbit_class"),
        $double("apogee_km").sub($double("perigee_km")).avg().as("mean_spread_km"))
      .sort("mean_spread_km", false));

## 5 · Group and count

The classic. `group(...)` then `agg(...)` with `$col(...))` and `count()`.

In [0]:
System.out.println(
    df.group("orbit_class").agg($col("orbit_class"), count()).sort("count", false));

## 6 · Sort

Sort a whole frame by a column, descending, and take the top.

In [0]:
System.out.println(
    df.sort("apogee_km", false)
      .cols("object_name", "apogee_km").select()
      .head(5));

In [0]:
// ── check ─────────────────────────────────────────────────────────────
var classes = df.group("orbit_class").agg($col("orbit_class"), count());
var pass = sel.height() == 21207 && geo.height() == 1183 && eccentric.height() == 569 && classes.height() == 4;
if (pass) {
    println("✓ select kept 21,207 rows · GEO 1,183 · ecc>0.25 569 · 4 orbit classes");
} else {
    println("✗ something is off — check the TODOs above");
}

## ★ BONUS — who owns the sky?

Fast finishers: the `satcat_owner` column is 104 distinct owners. Group, count,
sort, and take the top 10.

In [0]:
// ── ★ BONUS ───────────────────────────────────────────────────────────
System.out.println(
    df.group("satcat_owner").agg($col("satcat_owner"), count())
      .sort("count", false).head(10));

Next: [`04-plotting-with-gog4j.ipynb`](04-plotting-with-gog4j.ipynb).